# 08 — Balance Fitzpatrick17k Dataset (Augment Types 4/5/6)

Downloads all Fitzpatrick17k images from GCS, augments types 4/5/6
to 2500 each using 5 transform buckets at equal 20% proportion,
uploads all images (OG types 1-6 + augmented) to GCS bucket,
and saves CSV to Google Drive for manual AutoML.

**Augmentation buckets (20% each):**
| Bucket | Transform |
|--------|-----------|
| flip | HorizontalFlip |
| rotate | Rotate ±30° |
| crop | RandomResizedCrop |
| noise | GaussNoise |
| combined | All 4 together |

**Forbidden:** brightness, contrast, color jitter, blurring

In [ ]:
# Cell 1: Setup
import os, subprocess, sys

if not os.path.exists("/content/NST_Class"):
    subprocess.run(["git", "clone", "https://github.com/AayushBaniya2006/NST_Class.git"], cwd="/content")
else:
    subprocess.run(["git", "pull"], cwd="/content/NST_Class")
os.chdir("/content/NST_Class")

!pip install -q -r requirements.txt
!pip install -q albumentations

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/fitzpatrick17k.csv"):
    !wget -q -O data/fitzpatrick17k.csv "https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv"
    print("Downloaded fitzpatrick17k.csv")

sys.path.insert(0, "/content/NST_Class")
print("Setup complete!")

In [ ]:
# Cell 2: Authenticate with Google Cloud + mount Google Drive
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

!gcloud config set project gen-lang-client-0489710646
print("Authenticated + Drive mounted + project set!")

In [ ]:
# Cell 3: Configuration
import pandas as pd
import numpy as np

FITZ_BUCKET = "gs://fitzpatrick-dataset/all_images"
PROJECT_BUCKET = "augmentedbuckets"
GCS_IMAGE_PREFIX = f"gs://{PROJECT_BUCKET}/balanced_images"
LOCAL_IMAGE_DIR = "data/fitzpatrick_images"
LOCAL_AUG_DIR = "data/augmented_images"

# Google Drive output path
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NST_Class"

SEED = 42
IMAGE_SIZE = 224
TARGET_PER_CLASS = 2500
BUCKETS = ["flip", "rotate", "crop", "noise", "combined"]

# Load ALL Fitzpatrick17k metadata (types 1-6)
fitz_csv = pd.read_csv("data/fitzpatrick17k.csv")
fitz_csv["img_id"] = fitz_csv["md5hash"] + ".jpg"
fitz_csv = fitz_csv[fitz_csv["fitzpatrick_scale"].isin([1, 2, 3, 4, 5, 6])].copy()
df = fitz_csv[["img_id", "fitzpatrick_scale"]].copy()

print("Fitzpatrick17k — All 6 types:")
print(df["fitzpatrick_scale"].value_counts().sort_index())
print(f"\nTotal: {len(df)}")

# Augment ANY class below 2500
AUGMENTATION_TARGETS = {}
print(f"\nAugmentation plan (target {TARGET_PER_CLASS} minimum per class, 5 buckets @ 20% each):")
for cls in range(1, 7):
    current = len(df[df["fitzpatrick_scale"] == cls])
    needed = max(0, TARGET_PER_CLASS - current)
    if needed > 0:
        AUGMENTATION_TARGETS[cls] = needed
        per_bucket = needed // len(BUCKETS)
        remainder = needed % len(BUCKETS)
        print(f"  Type {cls}: {current} OG + {needed} aug = {current + needed} (AUGMENT)")
        print(f"           ({per_bucket} per bucket, +{remainder} extra in last bucket)")
    else:
        print(f"  Type {cls}: {current} OG (already >= {TARGET_PER_CLASS})")

In [ ]:
# Cell 4: Download ALL Fitzpatrick17k images from GCS (parallel + retry)
import time
import logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.cloud import storage
from tqdm import tqdm

# Suppress connection pool warnings
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

os.makedirs(LOCAL_IMAGE_DIR, exist_ok=True)

existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()} if os.path.isdir(LOCAL_IMAGE_DIR) else set()
needed_ids = set(df["img_id"]) - existing

if not needed_ids:
    print(f"All {len(df)} images already present locally")
else:
    print(f"Need {len(needed_ids)} images (have {len(existing)} already)")

    client = storage.Client()
    bucket_name = FITZ_BUCKET.replace("gs://", "").split("/")[0]
    prefix = "/".join(FITZ_BUCKET.replace("gs://", "").split("/")[1:])
    gcs_bucket = client.bucket(bucket_name)

    def _download_one(img_id):
        dest = os.path.join(LOCAL_IMAGE_DIR, img_id)
        if os.path.exists(dest):
            return "skip"
        blob_path = f"{prefix}/{img_id}" if prefix else img_id
        blob = gcs_bucket.blob(blob_path)
        for attempt in range(3):
            try:
                blob.download_to_filename(dest)
                return "ok"
            except Exception as e:
                if attempt == 2:
                    return f"fail: {e}"
                time.sleep(1)

    start = time.time()
    downloaded = 0
    failed = 0

    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(_download_one, img_id): img_id for img_id in needed_ids}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading"):
            result = future.result()
            if result == "ok":
                downloaded += 1
            elif result and result.startswith("fail"):
                failed += 1

    elapsed = time.time() - start
    print(f"\nDownloaded {downloaded}, failed {failed}, skipped {len(existing)} in {elapsed:.0f}s")

In [ ]:
# Cell 5: Verify downloads + recompute augmentation targets
from PIL import Image as PILImage
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

existing = {f.name for f in Path(LOCAL_IMAGE_DIR).iterdir() if f.is_file()}

def _check_image(img_id):
    if img_id not in existing:
        return None
    try:
        with PILImage.open(os.path.join(LOCAL_IMAGE_DIR, img_id)) as img:
            img.verify()
        return img_id
    except Exception:
        return False  # corrupt

present_ids = [i for i in df["img_id"] if i in existing]
available_ids = set()
corrupt = 0

with ThreadPoolExecutor(max_workers=16) as executor:
    for result in tqdm(executor.map(_check_image, present_ids), total=len(present_ids), desc="Validating"):
        if result is None:
            continue
        elif result is False:
            corrupt += 1
        else:
            available_ids.add(result)

df = df[df["img_id"].isin(available_ids)].copy()

print(f"\nValidation:")
print(f"  Available: {len(available_ids)}/{len(fitz_csv)}")
print(f"  Corrupt:   {corrupt}")
print(f"\nPer-class (usable):")
print(df["fitzpatrick_scale"].value_counts().sort_index())

# Recompute augmentation targets based on ACTUAL available images
AUGMENTATION_TARGETS = {}
print(f"\nFinal augmentation plan (target {TARGET_PER_CLASS} minimum per class):")
for cls in range(1, 7):
    current = len(df[df["fitzpatrick_scale"] == cls])
    needed = max(0, TARGET_PER_CLASS - current)
    if needed > 0:
        AUGMENTATION_TARGETS[cls] = needed
        per_bucket = needed // len(BUCKETS)
        remainder = needed % len(BUCKETS)
        print(f"  Type {cls}: {current} usable + {needed} to augment = {current + needed}")
    else:
        print(f"  Type {cls}: {current} usable (already >= {TARGET_PER_CLASS})")

In [ ]:
# Cell 6: Augment minority classes — 5 buckets at 20% each
from scripts.augment_minority import augment_images, verify_augmentation, TRANSFORM_BUCKETS

all_augmented = []

for fitz_class, total_needed in sorted(AUGMENTATION_TARGETS.items()):
    print(f"\n{'='*60}")
    print(f"Augmenting Fitzpatrick type {fitz_class} — {total_needed} images across 5 buckets")
    print(f"{'='*60}")

    class_imgs = df[df["fitzpatrick_scale"] == fitz_class]["img_id"].tolist()
    per_bucket = total_needed // len(BUCKETS)
    remainder = total_needed % len(BUCKETS)

    for i, bucket in enumerate(BUCKETS):
        # Last bucket gets the remainder
        bucket_count = per_bucket + (remainder if i == len(BUCKETS) - 1 else 0)
        if bucket_count == 0:
            continue

        bucket_dir = f"{LOCAL_AUG_DIR}/type_{fitz_class}/{bucket}"
        created, paths = augment_images(
            input_dir=LOCAL_IMAGE_DIR,
            output_dir=bucket_dir,
            img_ids=class_imgs,
            target_count=bucket_count,
            seed=SEED + fitz_class * 10 + i,  # unique seed per class+bucket
            image_size=IMAGE_SIZE,
            bucket=bucket,
        )

        print(f"  {bucket:>8}: {created}/{bucket_count} images created")

        # Verify each bucket
        result = verify_augmentation(LOCAL_IMAGE_DIR, bucket_dir)
        assert result["passed"], f"Verification failed for type {fitz_class} {bucket}: {result}"

        all_augmented.extend([
            {"img_id": p.name, "fitzpatrick_scale": fitz_class, "source": f"aug_{bucket}"}
            for p in paths
        ])

print(f"\n{'='*60}")
print(f"Total augmented: {len(all_augmented)}")
# Show breakdown by bucket
aug_df_temp = pd.DataFrame(all_augmented)
if len(aug_df_temp) > 0:
    print("\nPer-bucket breakdown:")
    print(aug_df_temp["source"].value_counts().sort_index())

In [ ]:
# Cell 7: Visual spot-check — one sample per bucket per class
import matplotlib.pyplot as plt
from PIL import Image as PILImage

n_classes = len(AUGMENTATION_TARGETS)
n_buckets = len(BUCKETS)
fig, axes = plt.subplots(n_classes, n_buckets, figsize=(3 * n_buckets, 3 * n_classes))
if n_classes == 1:
    axes = [axes]

for row, fitz_class in enumerate(sorted(AUGMENTATION_TARGETS.keys())):
    for col, bucket in enumerate(BUCKETS):
        ax = axes[row][col]
        bucket_dir = Path(f"{LOCAL_AUG_DIR}/type_{fitz_class}/{bucket}")
        aug_files = sorted(bucket_dir.glob("*"))
        if aug_files:
            ax.imshow(PILImage.open(aug_files[0]))
            ax.set_title(f"T{fitz_class} {bucket}", fontsize=8)
        ax.axis("off")

plt.suptitle("Augmented Spot-Check: 1 sample per bucket per class", fontsize=12)
plt.tight_layout()
plt.show()
plt.close()

print("If images look wrong, adjust parameters and re-run Cell 6.")

In [ ]:
# Cell 8: Build balanced CSV (all 6 types OG + augmented for 4/5/6)
df["source"] = "original"
aug_df = pd.DataFrame(all_augmented)
balanced_df = pd.concat([df, aug_df], ignore_index=True)

print("Balanced dataset (all types):")
print(balanced_df["fitzpatrick_scale"].value_counts().sort_index())
print(f"\nTotal: {len(balanced_df)}")
print(f"  Original (types 1-6): {len(df)}")
print(f"  Augmented (types 4-6): {len(aug_df)}")
print(f"\nPer-source breakdown:")
print(balanced_df["source"].value_counts().sort_index())

balanced_df.to_csv("balanced_dataset.csv", index=False)
print("\nSaved balanced_dataset.csv")

In [ ]:
# Cell 9: Upload all images to GCS + save CSV to Drive
import shutil

os.environ["CLOUDSDK_STORAGE_THREAD_COUNT"] = "32"
os.environ["CLOUDSDK_STORAGE_PROCESS_COUNT"] = "4"

# Copy augmented images into the main image dir so we upload everything once
aug_count = 0
for fitz_class in sorted(AUGMENTATION_TARGETS.keys()):
    for bucket in BUCKETS:
        bucket_dir = Path(f"{LOCAL_AUG_DIR}/type_{fitz_class}/{bucket}")
        if bucket_dir.is_dir():
            for f in bucket_dir.iterdir():
                if f.is_file():
                    shutil.copy2(str(f), os.path.join(LOCAL_IMAGE_DIR, f.name))
                    aug_count += 1
print(f"Copied {aug_count} augmented images into {LOCAL_IMAGE_DIR}")

# Upload all images (OG types 1-6 + augmented) to GCS
print("\nUploading all images to GCS (skips existing)...")
!gcloud storage cp -r "{LOCAL_IMAGE_DIR}/*" "{GCS_IMAGE_PREFIX}/"

# Upload CSV to GCS
!gcloud storage cp balanced_dataset.csv "gs://{PROJECT_BUCKET}/balanced_dataset.csv"
print("GCS upload complete!")

# Save CSV to Google Drive
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
drive_csv_path = f"{DRIVE_OUTPUT_DIR}/balanced_dataset.csv"
shutil.copy2("balanced_dataset.csv", drive_csv_path)
print(f"\nCSV saved to Google Drive: {drive_csv_path}")
print("You can now use this CSV for AutoML manually.")

In [ ]:
# Cell 9b: Generate AutoML import CSV + grant access to AutoML account
AUTOML_EMAIL = "pragmatesocialmedia@gmail.com"

# Build AutoML CSV: gs://path,label (no header)
automl_df = balanced_df[["img_id", "fitzpatrick_scale"]].copy()
automl_df["gcs_path"] = f"gs://{PROJECT_BUCKET}/balanced_images/" + automl_df["img_id"]
automl_df["label"] = automl_df["fitzpatrick_scale"].astype(str)
automl_csv = automl_df[["gcs_path", "label"]]
automl_csv.to_csv("automl_import.csv", index=False, header=False)
print(f"AutoML CSV: {len(automl_csv)} rows")
print(automl_csv.head())

# Upload AutoML CSV to bucket
!gcloud storage cp automl_import.csv gs://{PROJECT_BUCKET}/automl_import.csv

# Save to Drive too
import shutil
shutil.copy2("automl_import.csv", f"{DRIVE_OUTPUT_DIR}/automl_import.csv")

# Grant the AutoML email read access to the bucket
!gcloud storage buckets add-iam-policy-binding gs://{PROJECT_BUCKET} --member="user:{AUTOML_EMAIL}" --role="roles/storage.objectViewer"

# Grant the Vertex AI service account access (needed for AutoML to read images)
# Get the project number for the AutoML project
print(f"\n--- IMPORTANT ---")
print(f"You also need to grant the Vertex AI service account access.")
print(f"From the AutoML email's project, find the project number and run:")
print(f"  !gcloud storage buckets add-iam-policy-binding gs://{PROJECT_BUCKET} \\")
print(f'    --member="serviceAccount:service-PROJECT_NUMBER@gcp-sa-aiplatform.iam.gserviceaccount.com" \\')
print(f'    --role="roles/storage.objectViewer"')
print(f"\nOr run this if the AutoML project number is 167974988600:")
!gcloud storage buckets add-iam-policy-binding gs://{PROJECT_BUCKET} --member="serviceAccount:service-167974988600@gcp-sa-aiplatform.iam.gserviceaccount.com" --role="roles/storage.objectViewer"

print(f"\nDone! On the AutoML account, import: gs://{PROJECT_BUCKET}/automl_import.csv")

In [ ]:
# Cell 10: Summary
print("=" * 60)
print("AUGMENTATION SUMMARY")
print("=" * 60)
print(f"\nTarget per class: {TARGET_PER_CLASS} minimum")
print(f"Augmentation buckets: {', '.join(BUCKETS)} (20% each)")
print(f"\nPer-class distribution:")
for cls in range(1, 7):
    orig = len(df[df["fitzpatrick_scale"] == cls])
    total = len(balanced_df[balanced_df["fitzpatrick_scale"] == cls])
    aug = total - orig
    marker = " (augmented)" if cls in AUGMENTATION_TARGETS else ""
    print(f"  Type {cls}: {orig} original + {aug} augmented = {total}{marker}")
print(f"\nTotal images: {len(balanced_df)}")
print(f"  Original: {len(df)}")
print(f"  Augmented: {len(aug_df)}")
print(f"\nOutputs:")
print(f"  GCS images:  {GCS_IMAGE_PREFIX}/")
print(f"  GCS CSV:     gs://{PROJECT_BUCKET}/balanced_dataset.csv")
print(f"  Drive CSV:   {DRIVE_OUTPUT_DIR}/balanced_dataset.csv")
print(f"\nNext step: use the Drive CSV for AutoML manually.")
print("=" * 60)